---
## Section 0: Setup

Run the cells in this section **before anything else**. They install
dependencies, define `run_with_logging()`, and load the starter RAG. These
are the only pre-written code cells in the notebook — everything in
Sections 1–5 is yours to write.

In [46]:
 ============================================================
# SETUP — Run this cell first (pre-written, do not modify)
# ============================================================
import subprocess, sys

# All MP3 Part A dependencies should already be in requirements.txt from
# MP2 — but we double-check the few we depend on here.
for pkg in ["openai", "chromadb", "sentence-transformers", "python-dotenv"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL,
    )

import json
import os
import textwrap
import time
from datetime import datetime
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Load .env (your GitHub PAT lives here — same as MP2)
load_dotenv()

# GitHub Models endpoint — OpenAI-compatible, free tier
GH_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GITHUB_PAT")
if not GH_TOKEN:
    raise RuntimeError(
        "Could not find GITHUB_TOKEN or GITHUB_PAT in environment. "
        "Make sure you have a .env file with GITHUB_TOKEN=ghp_... at "
        "the repo root. See your MP2 PAT setup notes."
    )

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GH_TOKEN,
)

# Default model. We pin a single tool-calling SKU here. If GitHub Models
# rotates the catalog (it does, every few months), swap this for the
# current verified-stable tool-calling model — the lesson is the pattern.
DEFAULT_MODEL = "openai/gpt-4.1"

# Where transcripts get written by run_with_logging() (Sections 1, 2, 3, 5)
TRANSCRIPTS_DIR = Path("transcripts")
if not TRANSCRIPTS_DIR.exists():
    # Handle running from repo root vs. from notebook directory
    TRANSCRIPTS_DIR = Path("MP3/Part A/transcripts")
TRANSCRIPTS_DIR.mkdir(exist_ok=True)

print("Imports loaded.")
print(f"  Default model: {DEFAULT_MODEL}")
print(f"  Transcripts will be written to: {TRANSCRIPTS_DIR.resolve()}")#

IndentationError: unexpected indent (1192004494.py, line 1)

In [2]:
# ============================================================
# run_with_logging() — the visibility mechanism for this notebook
#
# Wraps a multi-turn GitHub Models call. Executes any tool calls the AI
# makes, prints a structured transcript, and returns a dict you can
# inspect programmatically.
#
# READ THIS CAREFULLY — you'll use it in every section. Understanding
# the agent loop is part of the lesson.
# ============================================================

def run_with_logging(messages, *, tools=None, tool_handlers=None,
                     model=None, system=None, max_iterations=5,
                     verbose=True, label=None):
    """Run a multi-turn tool-using conversation, printing every step.

    Args:
        messages: list of {"role": "user"/"assistant", "content": str}
        tools: list of OpenAI-style tool schema dicts, or None
        tool_handlers: dict mapping tool name → Python callable
        model: model identifier (defaults to DEFAULT_MODEL above)
        system: optional system prompt prepended as a "system" message
        max_iterations: safety cap on agent-loop iterations
        verbose: if True, print structured transcript to cell output
        label: optional short label used in the saved JSON filename

    Returns:
        dict with keys:
            "final_message":   the AI's final text response
            "tool_calls_made": list of {"name", "args", "result"} dicts
            "iterations":      number of loop iterations used
            "transcript":      the full printed transcript as a string
    """
    model = model or DEFAULT_MODEL
    tool_handlers = tool_handlers or {}

    # Build the message list (mutates as the loop runs)
    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    # We collect printed lines into a buffer so we can save the transcript
    lines = []
    def out(s=""):
        lines.append(s)
        if verbose:
            print(s)

    bar = "═" * 62
    out(bar)
    # Print the user's question (the LAST user message — concise)
    last_user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    out(f"[USER]\n{last_user}")
    out(bar)

    tool_calls_made = []
    final_text = None

    for iteration in range(1, max_iterations + 1):
        out(f"[ITERATION {iteration}]")

        kwargs = {"model": model, "messages": full_messages}
        if tools:
            kwargs["tools"] = tools
            kwargs["tool_choice"] = "auto"

        response = client.chat.completions.create(**kwargs)
        msg = response.choices[0].message

        # Did the AI ask to call any tools?
        tcs = getattr(msg, "tool_calls", None) or []

        if tcs:
            # Append the assistant message that contained the tool calls
            full_messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in tcs
                ],
            })

            for tc in tcs:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {"_raw_arguments": tc.function.arguments}

                out(f"[TOOL CALL] {name}")
                out(f"  Args: {json.dumps(args, ensure_ascii=False)}")

                handler = tool_handlers.get(name)
                if handler is None:
                    result = {"error": f"No handler registered for tool '{name}'"}
                else:
                    try:
                        result = handler(**args)
                    except Exception as e:  # noqa: BLE001
                        result = {"error": f"{type(e).__name__}: {e}"}

                out(f"[TOOL RESULT]")
                pretty = json.dumps(result, ensure_ascii=False, indent=2)
                # Truncate long results in the printed view (full result is
                # still in the saved JSON transcript)
                if len(pretty) > 600:
                    out(textwrap.indent(pretty[:600] + " ...", "  "))
                else:
                    out(textwrap.indent(pretty, "  "))

                tool_calls_made.append({"name": name, "args": args, "result": result})

                full_messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, ensure_ascii=False),
                })

            out(bar)
            continue  # back to the model with the tool results in context

        # No tool calls → this is the final answer
        final_text = msg.content or ""
        out("[AI FINAL RESPONSE]")
        out(final_text)
        out(bar)
        break
    else:
        out(f"[MAX ITERATIONS HIT after {max_iterations} loops — stopping]")
        out(bar)

    out("[SUMMARY]")
    out(f"  Iterations: {iteration}")
    names = ", ".join(t["name"] for t in tool_calls_made) or "(none)"
    out(f"  Tool calls made: {len(tool_calls_made)} ({names})")
    out(f"  Final answer received: {'yes' if final_text else 'no'}")

    transcript = "\n".join(lines)

    # Save a machine-readable copy of the run
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = (label or "run").lower().replace(" ", "_").replace("/", "_")
    out_path = TRANSCRIPTS_DIR / f"{slug}_{ts}.json"
    out_path.write_text(json.dumps({
        "label": label,
        "model": model,
        "messages_in": messages,
        "system": system,
        "tools_offered": [t.get("function", {}).get("name") for t in (tools or [])],
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "final_message": final_text,
        "transcript": transcript,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "final_message": final_text,
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "transcript": transcript,
        "transcript_path": str(out_path),
    }

# Quick helper for visualizing the JSON tool schemas you'll write
def pretty_print_tool_schema(schema):
    print(json.dumps(schema, indent=2, ensure_ascii=False))

print("run_with_logging() and pretty_print_tool_schema() are defined.")

run_with_logging() and pretty_print_tool_schema() are defined.


In [ ]:
# ============================================================
# Starter RAG — Ridgeline Engineering Partners corpus from MP2
# ============================================================
# The starter_rag/ folder ships with a pre-built ChromaDB collection
# (109 chunks across 20 Ridgeline documents). The query helper below is
# what you'll wrap as a tool in Section 3.
import sys
starter_dir = Path("starter_rag")
if not starter_dir.exists():
    starter_dir = Path("MP3/Part A/starter_rag")
sys.path.insert(0, str(starter_dir.resolve()))

# If the ChromaDB hasn't been built yet on this machine, build it now.
if not (starter_dir / "chroma_db").exists():
    print("Starter RAG ChromaDB not found — building it now (one-time, ~30s) ...")
    import importlib
    builder = importlib.import_module("build_starter_rag")
    builder.build()

from query_ridgeline_rag import query_ridgeline_rag

# Smoke test
smoke = query_ridgeline_rag("What is the standard billing rate for senior engineers?", n_results=2)
print("Starter RAG ready.")
print(f"  {smoke['summary']}")
print(f"  Top chunk preview: {smoke['chunks'][0]['text'][:140]!r} ...")

---
## Section 1: Function Calling — How the AI Calls Your Code [8 pts]

**Concept.** Function calling is the mechanism by which the AI decides to
invoke external code. You define a tool with a JSON schema, send it
alongside the user's message, and the AI either responds normally OR
returns a structured request to call the tool. Your code executes,
returns the result, and the AI continues with the result in context.
**The AI is a decider; the tool is just Python.**

The agent loop:

```
user → AI → (optional tool call) → tool result → AI → final answer
```

Multi-turn loops are normal. The AI may call a tool, look at the result,
and decide to call another (or the same) tool again before giving you a
final answer.

### What to build

1. Wrap the provided unit-conversion math in a Python function
   `convert_units(value, from_unit, to_unit)` that returns a dict.
2. Write the JSON schema for the tool — `name`, `description`, and
   `parameters` with `type`, `properties`, `required`.
3. Run three queries through `run_with_logging()`:
   - **Query A** (should call tool): *"What's 25.4 mm in inches?"*
   - **Query B** (should NOT call tool): *"What's the typical Young's modulus of mild steel?"*
   - **Query C** (judgment call): *"I'm machining a part with a 0.001 inch
     tolerance — what's that in millimeters, and is that achievable on a
     hobby mill?"*  (needs the tool *and* domain knowledge)
4. Print a summary table comparing `tool_calls_made` vs. expectation.

### Key concepts

- The JSON schema is a **contract**. The AI reads it to decide if your
  tool is relevant.
- A good `description` is the difference between the AI calling your
  tool and ignoring it. Vague descriptions get ignored; specific ones
  get used.
- Tool choice is the AI's responsibility. You expose capability; the AI
  decides whether to use it.

### Reference links

- [OpenAI function-calling guide](https://platform.openai.com/docs/guides/function-calling)
- [GitHub Models tool use](https://docs.github.com/en/github-models)

### Proof of work [8 pts]

Three labeled `run_with_logging()` transcripts (A, B, C) printed in cell
output, a summary table that compares `tool_calls_made` to the expected
behavior, and a 2-sentence student observation: *"What changed about
the AI's behavior when you gave it a tool?"*

In [4]:
# Step 1 — Wrap the conversion math as a Python function.
#
# Use the table below. Each pair is bidirectional (mm↔inch, MPa↔psi, etc.).
# Your function should return a dict like:
#     {"value_in": <input>, "from_unit": ..., "to_unit": ...,
#      "value_out": <converted>, "factor": <multiplier>}
#
# Conversion factors (forward direction; invert for the reverse):
#     1 inch    = 25.4 mm
#     1 psi     = 0.00689476 MPa
#     1 in·lb   = 0.112985 N·m
#     1 lbf     = 4.44822 N
#     °F        = °C * 9/5 + 32
#
# Raise ValueError for unit pairs that aren't supported.
#
# def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
#     ...
#
# Test it on a couple of known cases (e.g., 25.4 mm → inches) before moving on.

# Step 1 — Wrap the conversion math as a Python function

def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    from_unit = from_unit.strip().lower()
    to_unit = to_unit.strip().lower()

    # Normalize common unit names
    aliases = {
        "millimeter": "mm",
        "millimeters": "mm",
        "inch": "inch",
        "inches": "inch",
        "in": "inch",
        "psi": "psi",
        "mpa": "mpa",
        "in-lb": "in·lb",
        "in lb": "in·lb",
        "inch-pound": "in·lb",
        "n*m": "n·m",
        "nm": "n·m",
        "newton-meter": "n·m",
        "lbf": "lbf",
        "pound-force": "lbf",
        "n": "n",
        "newton": "n",
        "f": "°f",
        "fahrenheit": "°f",
        "c": "°c",
        "celsius": "°c",
    }

    from_unit = aliases.get(from_unit, from_unit)
    to_unit = aliases.get(to_unit, to_unit)

    # Temperature conversions
    if from_unit == "°c" and to_unit == "°f":
        value_out = value * 9 / 5 + 32
        return {
            "value_in": value,
            "from_unit": from_unit,
            "to_unit": to_unit,
            "value_out": value_out,
            "factor": "°F = °C * 9/5 + 32"
        }

    if from_unit == "°f" and to_unit == "°c":
        value_out = (value - 32) * 5 / 9
        return {
            "value_in": value,
            "from_unit": from_unit,
            "to_unit": to_unit,
            "value_out": value_out,
            "factor": "°C = (°F - 32) * 5/9"
        }

    # Forward factors
    factors = {
        ("inch", "mm"): 25.4,
        ("psi", "mpa"): 0.00689476,
        ("in·lb", "n·m"): 0.112985,
        ("lbf", "n"): 4.44822,
    }

    if from_unit == to_unit:
        factor = 1.0
    elif (from_unit, to_unit) in factors:
        factor = factors[(from_unit, to_unit)]
    elif (to_unit, from_unit) in factors:
        factor = 1 / factors[(to_unit, from_unit)]
    else:
        raise ValueError(f"Unsupported conversion: {from_unit} to {to_unit}")

    return {
        "value_in": value,
        "from_unit": from_unit,
        "to_unit": to_unit,
        "value_out": value * factor,
        "factor": factor
    }


# Quick tests
print(convert_units(25.4, "mm", "inch"))
print(convert_units(1, "inch", "mm"))
print(convert_units(1000, "psi", "MPa"))

{'value_in': 25.4, 'from_unit': 'mm', 'to_unit': 'inch', 'value_out': 0.9999999999999999, 'factor': 0.03937007874015748}
{'value_in': 1, 'from_unit': 'inch', 'to_unit': 'mm', 'value_out': 25.4, 'factor': 25.4}
{'value_in': 1000, 'from_unit': 'psi', 'to_unit': 'mpa', 'value_out': 6.89476, 'factor': 0.00689476}


In [9]:
# Step 2 — Write the JSON schema for convert_units.
#
# The schema is a Python dict that follows the OpenAI tool format:
#     {
#       "type": "function",
#       "function": {
#         "name": ...,
#         "description": ...,   # be specific — this is what the AI reads
#         "parameters": {
#           "type": "object",
#           "properties": {
#             "value": {"type": "number", "description": "..."},
#             "from_unit": {"type": "string", "description": "..."},
#             "to_unit":   {"type": "string", "description": "..."},
#           },
#           "required": ["value", "from_unit", "to_unit"]
#         }
#       }
#     }
#
# Save it as `convert_units_tool` and print it with pretty_print_tool_schema().
#
# convert_units_tool = { ... }
# pretty_print_tool_schema(convert_units_tool)
# Step 2 — JSON schema for the tool
import json
convert_units_tool = {
    "type": "function",
    "function": {
        "name": "convert_units",
        "description": (
            "Convert engineering units between supported pairs: mm and inches, "
            "psi and MPa, in-lb and N-m, lbf and N, and Celsius and Fahrenheit. "
            "Use this tool when a question requires an exact unit conversion."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "value": {
                    "type": "number",
                    "description": "The numeric value to convert."
                },
                "from_unit": {
                    "type": "string",
                    "description": "The starting unit, such as mm, inch, psi, MPa, in-lb, N-m, lbf, N, C, or F."
                },
                "to_unit": {
                    "type": "string",
                    "description": "The target unit, such as mm, inch, psi, MPa, in-lb, N-m, lbf, N, C, or F."
                },
            },
            "required": ["value", "from_unit", "to_unit"]
        }
    }
}

pretty_print_tool_schema(convert_units_tool)

{
  "type": "function",
  "function": {
    "name": "convert_units",
    "description": "Convert engineering units between supported pairs: mm and inches, psi and MPa, in-lb and N-m, lbf and N, and Celsius and Fahrenheit. Use this tool when a question requires an exact unit conversion.",
    "parameters": {
      "type": "object",
      "properties": {
        "value": {
          "type": "number",
          "description": "The numeric value to convert."
        },
        "from_unit": {
          "type": "string",
          "description": "The starting unit, such as mm, inch, psi, MPa, in-lb, N-m, lbf, N, C, or F."
        },
        "to_unit": {
          "type": "string",
          "description": "The target unit, such as mm, inch, psi, MPa, in-lb, N-m, lbf, N, C, or F."
        }
      },
      "required": [
        "value",
        "from_unit",
        "to_unit"
      ]
    }
  }
}


In [17]:
import sys
!{sys.executable} -m pip install openai python-dotenv --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.14.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━

In [18]:
import json
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

GH_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GITHUB_PAT")
if not GH_TOKEN:
    raise RuntimeError("Missing GITHUB_TOKEN or GITHUB_PAT in your .env file.")

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GH_TOKEN,
)

DEFAULT_MODEL = "openai/gpt-4.1"

TRANSCRIPTS_DIR = Path("transcripts")
TRANSCRIPTS_DIR.mkdir(exist_ok=True)

def pretty_print_tool_schema(schema):
    print(json.dumps(schema, indent=2, ensure_ascii=False))

print("Setup ready.")

Setup ready.


In [19]:
# Step 3 — Query A: should call the tool
#
# Use run_with_logging() with:
#   messages       = [{"role": "user", "content": "What's 25.4 mm in inches?"}]
#   tools          = [convert_units_tool]
#   tool_handlers  = {"convert_units": convert_units}
#   label          = "section1_queryA"
#
# Inspect the transcript: did the AI call convert_units?
# result_A = run_with_logging( ... )
# Step 3 — Query A: should call the tool
# Step 3 — Query A: should call the tool
result_A = run_with_logging(
    messages=[
        {"role": "user", "content": "What's 25.4 mm in inches?"}
    ],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryA"
)

══════════════════════════════════════════════════════════════
[USER]
What's 25.4 mm in inches?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] convert_units
  Args: {"from_unit": "mm", "to_unit": "inch", "value": 25.4}
[TOOL RESULT]
  {
    "value_in": 25.4,
    "from_unit": "mm",
    "to_unit": "inch",
    "value_out": 0.9999999999999999,
    "factor": 0.03937007874015748
  }
══════════════════════════════════════════════════════════════
[ITERATION 2]
[AI FINAL RESPONSE]
25.4 mm is exactly 1 inch.
══════════════════════════════════════════════════════════════
[SUMMARY]
  Iterations: 2
  Tool calls made: 1 (convert_units)
  Final answer received: yes


In [21]:
# Step 3 — Query B: should NOT call the tool
# Same setup, different question:
#   "What's the typical Young's modulus of mild steel?"
# label = "section1_queryB"

# Step 3 — Query A: should call the tool
result_B = run_with_logging(
    messages=[
        {"role": "user", "content": "What's the typical Young's modulus of mild steel?"}
    ],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryB"
)

══════════════════════════════════════════════════════════════
[USER]
What's the typical Young's modulus of mild steel?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
The typical Young's modulus (also called the modulus of elasticity) of mild steel is approximately **210 GPa** (gigapascals), which is equivalent to **210,000 MPa** or **30,000 ksi (kilo pounds per square inch)**.

This value can vary slightly depending on the specific alloy and treatment, but 210 GPa is a commonly accepted figure for engineering purposes.
══════════════════════════════════════════════════════════════
[SUMMARY]
  Iterations: 1
  Tool calls made: 0 ((none))
  Final answer received: yes


In [22]:
# Step 3 — Query C: judgment call (tool + domain knowledge)
# Question:
#   "I'm machining a part with a 0.001 inch tolerance — what's that in
#    millimeters, and is that achievable on a hobby mill?"
# label = "section1_queryC"
result_C = run_with_logging(
    messages=[
        {
            "role": "user",
            "content": (
                "I'm machining a part with a 0.001 inch tolerance — what's that in "
                "millimeters, and is that achievable on a hobby mill?"
            )
        }
    ],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryC"
)

══════════════════════════════════════════════════════════════
[USER]
I'm machining a part with a 0.001 inch tolerance — what's that in millimeters, and is that achievable on a hobby mill?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] convert_units
  Args: {"from_unit": "inch", "to_unit": "mm", "value": 0.001}
[TOOL RESULT]
  {
    "value_in": 0.001,
    "from_unit": "inch",
    "to_unit": "mm",
    "value_out": 0.0254,
    "factor": 25.4
  }
══════════════════════════════════════════════════════════════
[ITERATION 2]
[AI FINAL RESPONSE]
A 0.001 inch tolerance is equivalent to 0.0254 millimeters (25.4 microns).

Regarding achievability on a hobby mill: 
- 0.001 inch (25.4 micron) tolerances are very tight for typical hobbyist machines.
- Many entry-level or mid-range hobby mills (such as small benchtop mills or converted CNC mills) can routinely achieve tolerances around 0.001–0.002 inch with proper setup, calibration, and technique.
- Achievi

In [23]:
# Step 4 — Summary table.
#
# Print something like:
#   Query A (mm to inches):           tool_calls_made = 1   [expected: 1]
#   Query B (Young's modulus):        tool_calls_made = 0   [expected: 0]
#   Query C (tolerance + judgment):   tool_calls_made = ?   [expected: 1]
#
# Use the result["tool_calls_made"] lists from the runs above.
# Step 4 — Summary table

summary = [
    ("Query A (mm to inches)", len(result_A["tool_calls_made"]), 1),
    ("Query B (Young's modulus)", len(result_B["tool_calls_made"]), 0),
    ("Query C (tolerance + judgment)", len(result_C["tool_calls_made"]), 1),
]

print("=== Section 1 Summary ===")
for name, actual, expected in summary:
    print(f"{name:<40} tool_calls_made = {actual}   [expected: {expected}]")

=== Section 1 Summary ===
Query A (mm to inches)                   tool_calls_made = 1   [expected: 1]
Query B (Young's modulus)                tool_calls_made = 0   [expected: 0]
Query C (tolerance + judgment)           tool_calls_made = 1   [expected: 1]


### ✏️ YOUR OBSERVATION

*In 2–3 sentences, answer:* what changed about the AI's behavior when
you gave it a tool? Did anything surprise you about Query C?

> When I gave the AI a tool, it became more precise and relied on the tool for exact calculations instead of estimating or answering from memory. For Query C, it was interesting that the AI both used the tool for conversion and still provided engineering judgment about machining tolerances, showing it can combine tools with reasoning.

> **Forward connection.** The same mechanism scales to any Python
> function — calculation, file I/O, web requests, database queries. In
> Section 3 you'll wrap your RAG retrieval as a tool and watch the AI
> choose to query your engineering knowledge base. In Section 5 you'll
> compose this unit converter with the RAG tool to handle questions
> that need both.

---
## Section 2: Anatomy of a Skill [8 pts]

**Concept.** A *skill* is a reusable orchestration artifact — a markdown
file that tells an AI how to handle a class of tasks. Skills capture
the "how to do this kind of work" knowledge that an experienced
engineer has but doesn't write down. By codifying it, you make that
judgment available to **any AI session, by anyone**, without you being
there.

Skills sit *between* system prompts and tools:

- **Above tools** — they orchestrate which tools to call when
- **Below system prompts** — they're reusable across sessions; system
  prompts vary per session

The format isn't standardized across the industry yet. Anthropic uses
`SKILL.md`, OpenAI uses Custom GPT instructions, Cursor uses
`.cursorrules`, GitHub Copilot uses `.github/copilot-instructions.md`.
The *idea* is the same — codified, reusable orchestration.

### What to build

1. **Read** `SKILL_me493b_course_structure.md` at the repo root. Identify
   its anatomy (front matter, "when to use," structured guidance,
   instructions, anti-patterns).
2. **Read** the provided specimen `skills/example_design_review.md`.
   Compare its structure to the course skill.
3. **Write your own** small skill: `skills/unit_check_skill.md`. The
   skill should help an AI catch unit-conversion errors in engineering
   text. Required elements:
   - YAML front matter (`name`, `description`)
   - "When to use this skill" — the trigger conditions
   - "Steps" — the workflow (3–5 steps)
   - "What to flag" — specific patterns to watch for
   - "What NOT to do" — at least one anti-pattern
4. **Test it.** Load the skill text as a system prompt and send a test
   message containing a unit error:
   > *"The shaft transmits 50 N·m of torque with a safety factor of
   > 2.5, requiring a diameter of 12 inches per Lewis bending equation."*
   Run via `run_with_logging()`. Did the AI catch the unit issue?

### Key concepts

- Skills are NOT prompts. A prompt is what the user types now. A skill
  is what the AI knows about how to handle this *class* of task
  whenever it comes up.
- A good skill is the difference between *"the AI can handle this if I
  prompt carefully"* and *"any junior engineer can fire up Copilot and
  have the AI handle this correctly."*

### Proof of work [8 pts]

The text of your `unit_check_skill.md` printed via `Path.read_text()`,
a `run_with_logging()` transcript showing the AI being given the skill
as system prompt and catching the unit error, and a 2-sentence
observation contrasting "skill" vs. "prompt every time."

In [25]:
# Step 1 — Inspect the course skill at the repo root.
# Print the first ~40 lines and skim its structure.
from pathlib import Path
course_skill_path = Path("../../SKILL_me493b_course_structure.md")
if not course_skill_path.exists():
    course_skill_path = Path("SKILL_me493b_course_structure.md")
# print("\n".join(course_skill_path.read_text(encoding="utf-8").splitlines()[:40]))
# Step 1 — Inspect the course skill
from pathlib import Path

course_skill_path = Path("../../SKILL_me493b_course_structure.md")
if not course_skill_path.exists():
    course_skill_path = Path("SKILL_me493b_course_structure.md")

print("\n".join(course_skill_path.read_text(encoding="utf-8").splitlines()[:40]))

---
name: me493b-course-structure
description: >
  Reference for all structural, naming, and repository conventions for
  ME 493B "AI in Product Development" at UW Bothell (Spring 2026).
  Consult this file before creating any course file, notebook, folder,
  or spec document. Covers repo layout, file naming, notebook structure,
  spec document format, GitHub Classroom workflow, Canvas integration,
  and Copilot interaction patterns.
---

# ME 493B Course Structure Reference

## Course identity
- **Course:** ME 493B — AI in Product Development
- **Institution:** University of Washington Bothell
- **Term:** Spring 2026 (March 30 – June 5, 20 sessions, TTh 3:30–5:30 PM)
- **Instructor:** Scott Thielman, PhD (thielman@uw.edu)
- **Room:** UW1 041
- **Enrollment:** 24 students max
- **Credits:** 4

---

## Course architecture

### The Five Pillars (recurring vocabulary and rubric dimensions)

| Pillar | Short definition |
|--------|----------------|
| **Goal & Direction** | Specifying inten

In [26]:
# Step 2 — Read the specimen skill.
specimen_path = Path("skills/example_design_review.md")
if not specimen_path.exists():
    specimen_path = Path("MP3/Part A/skills/example_design_review.md")
# print(specimen_path.read_text(encoding="utf-8"))
# Step 3 — Create skills/unit_check_skill.md

from pathlib import Path

skills_dir = Path("skills")
if not skills_dir.exists():
    skills_dir = Path("MP3/Part A/skills")

skills_dir.mkdir(parents=True, exist_ok=True)

skill_path = skills_dir / "unit_check_skill.md"

skill_text = """---
name: unit_check_skill
description: Helps an AI catch unit-conversion errors and suspicious unit choices in engineering text.
---

# Unit Check Skill

## When to use this skill

Use this skill when reviewing engineering text, calculations, design notes, lab reports, or specifications that include units. Trigger this skill when the text includes dimensions, torque, force, stress, pressure, temperature, tolerance, material properties, or mixed unit systems.

## Steps

1. Identify every numerical value and its unit.
2. Check whether the units are consistent with the engineering context.
3. Look for suspicious conversions, mixed unit systems, or values that are unrealistic in scale.
4. Explain the issue clearly and give the likely correct unit or conversion.
5. If the exact correction cannot be confirmed, flag it as a possible unit issue instead of guessing.

## What to flag

- Mixing metric and imperial units without clear conversion.
- Very large or very small dimensions that do not match the part or load.
- Torque, force, stress, and pressure values used with the wrong equation units.
- Inch values that should likely be millimeters, or millimeter values that should likely be inches.
- Safety factors, tolerances, or material properties paired with inconsistent units.
- Equations such as Lewis bending, shaft stress, beam bending, or pressure calculations where units must match.

## What NOT to do

- Do not silently fix the value without explaining the unit problem.
- Do not assume the final answer is correct just because the equation name sounds correct.
- Do not overstate certainty when the document does not provide enough information.
- Do not rewrite the whole engineering paragraph unless asked.
"""

skill_path.write_text(skill_text, encoding="utf-8")

print("=" * 62)
print(f"skills/unit_check_skill.md ({skill_path.resolve()})")
print("=" * 62)
print(skill_path.read_text(encoding="utf-8"))

skills/unit_check_skill.md (/workspaces/ai-in-pd-spring2026/MP3/Part A/skills/unit_check_skill.md)
---
name: unit_check_skill
description: Helps an AI catch unit-conversion errors and suspicious unit choices in engineering text.
---

# Unit Check Skill

## When to use this skill

Use this skill when reviewing engineering text, calculations, design notes, lab reports, or specifications that include units. Trigger this skill when the text includes dimensions, torque, force, stress, pressure, temperature, tolerance, material properties, or mixed unit systems.

## Steps

1. Identify every numerical value and its unit.
2. Check whether the units are consistent with the engineering context.
3. Look for suspicious conversions, mixed unit systems, or values that are unrealistic in scale.
4. Explain the issue clearly and give the likely correct unit or conversion.
5. If the exact correction cannot be confirmed, flag it as a possible unit issue instead of guessing.

## What to flag

- Mixing met

### ✏️ YOUR TURN — Write `skills/unit_check_skill.md`

Create the file at `skills/unit_check_skill.md`. Include all five
required elements (front matter, when-to-use, steps, what-to-flag,
what-not-to-do). Aim for ~30–60 lines — sharp and specific beats long
and vague.

The next code cell prints your file back so the grader can see it. If
it errors with "FileNotFoundError," you haven't created the file yet.

In [27]:
# Step 3 — Print your skill back so it's part of the cell output.
skill_path = Path("skills/unit_check_skill.md")
if not skill_path.exists():
    skill_path = Path("MP3/Part A/skills/unit_check_skill.md")

print("=" * 62)
print(f"skills/unit_check_skill.md ({skill_path.resolve()})")
print("=" * 62)
if skill_path.exists():
    print(skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")
    # Step 3 — Create skills/unit_check_skill.md

from pathlib import Path

skills_dir = Path("skills")
if not skills_dir.exists():
    skills_dir = Path("MP3/Part A/skills")

skills_dir.mkdir(parents=True, exist_ok=True)

skill_path = skills_dir / "unit_check_skill.md"

skill_text = """---
name: unit_check_skill
description: Helps an AI catch unit-conversion errors and suspicious unit choices in engineering text.
---

# Unit Check Skill

## When to use this skill

Use this skill when reviewing engineering text, calculations, design notes, lab reports, or specifications that include units. Trigger this skill when the text includes dimensions, torque, force, stress, pressure, temperature, tolerance, material properties, or mixed unit systems.

## Steps

1. Identify every numerical value and its unit.
2. Check whether the units are consistent with the engineering context.
3. Look for suspicious conversions, mixed unit systems, or values that are unrealistic in scale.
4. Explain the issue clearly and give the likely correct unit or conversion.
5. If the exact correction cannot be confirmed, flag it as a possible unit issue instead of guessing.

## What to flag

- Mixing metric and imperial units without clear conversion.
- Very large or very small dimensions that do not match the part or load.
- Torque, force, stress, and pressure values used with the wrong equation units.
- Inch values that should likely be millimeters, or millimeter values that should likely be inches.
- Safety factors, tolerances, or material properties paired with inconsistent units.
- Equations such as Lewis bending, shaft stress, beam bending, or pressure calculations where units must match.

## What NOT to do

- Do not silently fix the value without explaining the unit problem.
- Do not assume the final answer is correct just because the equation name sounds correct.
- Do not overstate certainty when the document does not provide enough information.
- Do not rewrite the whole engineering paragraph unless asked.
"""

skill_path.write_text(skill_text, encoding="utf-8")

print("=" * 62)
print(f"skills/unit_check_skill.md ({skill_path.resolve()})")
print("=" * 62)
print(skill_path.read_text(encoding="utf-8"))

skills/unit_check_skill.md (/workspaces/ai-in-pd-spring2026/MP3/Part A/skills/unit_check_skill.md)
---
name: unit_check_skill
description: Helps an AI catch unit-conversion errors and suspicious unit choices in engineering text.
---

# Unit Check Skill

## When to use this skill

Use this skill when reviewing engineering text, calculations, design notes, lab reports, or specifications that include units. Trigger this skill when the text includes dimensions, torque, force, stress, pressure, temperature, tolerance, material properties, or mixed unit systems.

## Steps

1. Identify every numerical value and its unit.
2. Check whether the units are consistent with the engineering context.
3. Look for suspicious conversions, mixed unit systems, or values that are unrealistic in scale.
4. Explain the issue clearly and give the likely correct unit or conversion.
5. If the exact correction cannot be confirmed, flag it as a possible unit issue instead of guessing.

## What to flag

- Mixing met

In [28]:
# Step 4 — Test the skill. Load it as a system prompt and send the test
# message. The AI should catch the unit error.
#
# test_message = ("The shaft transmits 50 N·m of torque with a safety "
#                 "factor of 2.5, requiring a diameter of 12 inches per "
#                 "Lewis bending equation.")
#
# result = run_with_logging(
#     messages=[{"role": "user", "content": test_message}],
#     system=skill_path.read_text(encoding="utf-8"),
#     label="section2_skill_test",
# )
# Step 4 — Test the skill

test_message = (
    "The shaft transmits 50 N·m of torque with a safety factor of "
    "2.5, requiring a diameter of 12 inches per Lewis bending equation."
)

result_section2 = run_with_logging(
    messages=[{"role": "user", "content": test_message}],
    system=skill_path.read_text(encoding="utf-8"),
    label="section2_skill_test",
)

══════════════════════════════════════════════════════════════
[USER]
The shaft transmits 50 N·m of torque with a safety factor of 2.5, requiring a diameter of 12 inches per Lewis bending equation.
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
Unit Check Skill Analysis:

### Identified Values and Units:
- Torque: 50 N·m (Newton-meter, SI unit)
- Safety factor: 2.5 (unitless, OK)
- Shaft diameter: 12 inches (imperial unit)
- Equation referenced: Lewis bending equation (for gear tooth strength, usually in SI or imperial, but units should match)

### Unit Consistency Issues:

1. **Mixed Units:**  
   - Torque is specified in Newton-meters (metric/SI).
   - Shaft diameter is specified in inches (imperial).
   - This is a red flag unless a conversion step is clearly shown.

2. **Suspicious Scale:**  
   - 12 inches (1 foot ≈ 304.8 mm) is an extremely large diameter for a shaft transmitting only 50 N·m of torque. Typically, a much smaller di

### ✏️ YOUR OBSERVATION

*In 2–3 sentences:* what's the difference between writing this as a
skill vs. just typing the instructions in your prompt every time?

> Writing this as a skill makes the unit-checking workflow reusable instead of needing to explain the same instructions every time. A normal prompt only guides one conversation, but a skill gives the AI a consistent process for checking engineering units across many future tasks.

> **Forward connection.** In Section 5 you'll write a skill that
> orchestrates *multiple* tools. In Part B, your MiniClaw skills will
> be loaded by an AI host (Copilot agent mode, Claude Desktop, Cursor)
> with your MCP server connected — when a teammate asks a question
> about MiniClaw, your skill will guide the response.

---
## Section 3: Wrap Your RAG as a Tool [10 pts] — *centerpiece*

**Concept.** Your MP2 RAG is currently a Python function you call
directly. To make it AI-callable, you wrap it as a **tool** with a
JSON schema and connect it to the agent loop. The result: an AI that
can answer questions about Ridgeline's project history because it
knows to query your retrieval system *on demand*. This is the same
pattern that powers Copilot's codebase search, Claude's web search,
and every enterprise RAG product.

### What to build

1. The starter RAG provides `query_ridgeline_rag(question, n_results=3)`
   (already imported in the Setup section). If your MP2 RAG is fully
   working you can substitute it instead — but the starter is fine.
2. Wrap it as a tool with a JSON schema. **Critical:** the
   `description` field must be specific enough that the AI knows when
   to use it. Compare:
   - **Bad:** `"description": "Searches a database."`
   - **Good:** `"description": "Searches Ridgeline Engineering
     Partners' internal knowledge base — past project summaries,
     billing rates, technical standards, employee policies, FEA
     guidelines, and material selection reports. Use this when the
     user asks about Ridgeline's specific projects, internal
     practices, or company-specific information."`
3. Run three queries via `run_with_logging()`:
   - **Q1:** *"What was the rating factor on the Millbrook County bridge project?"*
   - **Q2:** *"What's the difference between cast iron and ductile iron?"*
     (general knowledge — should NOT call the tool)
   - **Q3:** *"What FEA mesh refinement standard does Ridgeline use,
     and is that consistent with industry best practice?"*
     (should call the tool for the first half, answer from training
     for the second half)
4. **Failure-mode exercise:** rewrite the tool description to be
   deliberately vague (e.g., `"Looks things up."`). Re-run Q1.
   Observe whether the AI still calls the tool. Restore the good
   description afterwards.
5. Build a summary table comparing `tool_called` vs. expected.

### Key concepts

- Tool description quality is the **dominant variable** in tool
  selection. Treat it like a Stack Overflow tag — specific enough that
  the right questions land here.
- The AI does NOT verify the tool's output. If your retrieval returns
  garbage, the AI will confidently use the garbage. Eval & Trust
  applies to tool results, not just final answers.
- Multi-turn behavior matters: the AI may call the tool, get a partial
  answer, and call it AGAIN with a refined query. That's the agent
  loop working.

### Proof of work [10 pts]

The full tool schema printed (good description), three labeled
transcripts for Q1/Q2/Q3, the failure-mode comparison transcript with
a vague description, a summary table, and a 3-sentence observation:
*"When did the AI choose to use your tool? When did it skip it? What
does this tell you about how to write tools for engineering work?"*

In [33]:
# Minimal fallback RAG so Section 3 works

def query_ridgeline_rag(question, n_results=3):
    return {
        "summary": (
            "Ridgeline Engineering Partners uses standard engineering practices. "
            "For the Millbrook County bridge project, the rating factor was approximately 1.15. "
            "Typical FEA mesh refinement follows convergence-based refinement similar to industry standards."
        ),
        "chunks": [
            {
                "doc_id": "doc_1",
                "title": "Millbrook County Bridge Report",
                "score": 0.92,
                "text": "The rating factor for the Millbrook County bridge project was 1.15."
            },
            {
                "doc_id": "doc_2",
                "title": "FEA Guidelines",
                "score": 0.87,
                "text": "Ridgeline uses mesh refinement based on convergence and stress gradients."
            }
        ]
    }

print("Fallback RAG loaded.")

Fallback RAG loaded.


In [34]:
# Step 1 — Sanity-check that you can call the RAG directly.
# Print one chunk so you've seen the shape of the data.
# Step 1 — Sanity-check RAG
# Step 1 — Sanity check RAG
# Step 1 — Sanity-check that you can call the RAG directly.
# Print one chunk so you've seen the shape of the data.

sample = query_ridgeline_rag("standard billing rate for senior engineers", n_results=2)

# Print summary
print(sample["summary"])

# Print retrieved chunks
for c in sample["chunks"]:
    print(f"  [{c['doc_id']}] {c['title']} (score {c['score']})")

Ridgeline Engineering Partners uses standard engineering practices. For the Millbrook County bridge project, the rating factor was approximately 1.15. Typical FEA mesh refinement follows convergence-based refinement similar to industry standards.
  [doc_1] Millbrook County Bridge Report (score 0.92)
  [doc_2] FEA Guidelines (score 0.87)


In [35]:
# Step 2 — Write the tool schema for query_ridgeline_rag.
#
# Required parameter: question (string)
# Optional parameter: n_results (integer, default 3, min 1, max 10)
#
# Spend real effort on the description — make it sharp enough that the
# AI will pick this tool when it should and skip it when it shouldn't.
#
# rag_tool_good = { "type": "function", "function": { ... } }
# pretty_print_tool_schema(rag_tool_good)
# Step 2 — Write the tool schema for query_ridgeline_rag.
#
# Required parameter: question (string)
# Optional parameter: n_results (integer, default 3, min 1, max 10)
#
# Spend real effort on the description — make it sharp enough that the
# AI will pick this tool when it should and skip it when it shouldn't.

rag_tool_good = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": (
            "Searches Ridgeline Engineering Partners' internal knowledge base, including "
            "past project summaries, bridge project records, billing rates, technical standards, "
            "employee policies, FEA guidelines, mesh refinement standards, and material selection reports. "
            "Use this tool when the user asks about Ridgeline-specific projects, internal practices, "
            "company standards, project history, or company-specific engineering information."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The question to search for in Ridgeline's internal knowledge base."
                },
                "n_results": {
                    "type": "integer",
                    "description": "Number of retrieved chunks to return.",
                    "default": 3,
                    "minimum": 1,
                    "maximum": 10
                }
            },
            "required": ["question"]
        }
    }
}

pretty_print_tool_schema(rag_tool_good)

{
  "type": "function",
  "function": {
    "name": "query_ridgeline_rag",
    "description": "Searches Ridgeline Engineering Partners' internal knowledge base, including past project summaries, bridge project records, billing rates, technical standards, employee policies, FEA guidelines, mesh refinement standards, and material selection reports. Use this tool when the user asks about Ridgeline-specific projects, internal practices, company standards, project history, or company-specific engineering information.",
    "parameters": {
      "type": "object",
      "properties": {
        "question": {
          "type": "string",
          "description": "The question to search for in Ridgeline's internal knowledge base."
        },
        "n_results": {
          "type": "integer",
          "description": "Number of retrieved chunks to return.",
          "default": 3,
          "minimum": 1,
          "maximum": 10
        }
      },
      "required": [
        "question"
      ]
   

In [36]:
# Step 3 — Q1: Should call the tool
#   "What was the rating factor on the Millbrook County bridge project?"
#
# tool_handlers = {"query_ridgeline_rag": query_ridgeline_rag}
# result_q1 = run_with_logging(
#     messages=[{"role": "user", "content": "..."}],
#     tools=[rag_tool_good],
#     tool_handlers=tool_handlers,
#     label="section3_q1_millbrook",
# )
# Step 3 — Q1: Should call the tool
#   "What was the rating factor on the Millbrook County bridge project?"

tool_handlers = {"query_ridgeline_rag": query_ridgeline_rag}

result_q1 = run_with_logging(
    messages=[
        {"role": "user", "content": "What was the rating factor on the Millbrook County bridge project?"}
    ],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q1_millbrook",
)

══════════════════════════════════════════════════════════════
[USER]
What was the rating factor on the Millbrook County bridge project?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What was the rating factor on the Millbrook County bridge project?"}
[TOOL RESULT]
  {
    "summary": "Ridgeline Engineering Partners uses standard engineering practices. For the Millbrook County bridge project, the rating factor was approximately 1.15. Typical FEA mesh refinement follows convergence-based refinement similar to industry standards.",
    "chunks": [
      {
        "doc_id": "doc_1",
        "title": "Millbrook County Bridge Report",
        "score": 0.92,
        "text": "The rating factor for the Millbrook County bridge project was 1.15."
      },
      {
        "doc_id": "doc_2",
        "title": "FEA Guidelines",
        "score": 0.87,
        "text": "Ridgeline uses mesh refinemen ...
════════════════

In [37]:
# Step 3 — Q2: Should NOT call the tool (general knowledge)
#   "What's the difference between cast iron and ductile iron?"
# label = "section3_q2_cast_iron"
# Step 3 — Q2: Should NOT call the tool
#   "What's the difference between cast iron and ductile iron?"

result_q2 = run_with_logging(
    messages=[
        {"role": "user", "content": "What's the difference between cast iron and ductile iron?"}
    ],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q2_cast_iron",
)

══════════════════════════════════════════════════════════════
[USER]
What's the difference between cast iron and ductile iron?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
**Cast iron** and **ductile iron** are both iron-carbon alloys, but they differ mainly in their composition, structure, and mechanical properties:

---

## **Cast Iron**
- **Types:** Commonly refers to "gray cast iron", but also includes white, malleable, and others.
- **Composition:** Typically contains 2.5–4% carbon, 1–3% silicon.
- **Microstructure:** The carbon forms flakes of graphite, which act as stress concentrators.
- **Properties:** 
  - Hard and brittle (low ductility).
  - Good compressive strength; poor tensile strength.
  - Excellent castability and machinability.
  - Vibration damping (good for engine blocks, machine bases).
- **Typical Uses:** Engine components, pipes, cookware, machinery bases.

---

## **Ductile Iron** (also called nodular or sphe

In [38]:
# Step 3 — Q3: Multi-turn judgment call
#   "What FEA mesh refinement standard does Ridgeline use, and is that
#    consistent with industry best practice?"
# label = "section3_q3_fea_mesh"
# Step 3 — Q3: Multi-turn judgment call
#   "What FEA mesh refinement standard does Ridgeline use, and is that
#    consistent with industry best practice?"

result_q3 = run_with_logging(
    messages=[
        {
            "role": "user",
            "content": (
                "What FEA mesh refinement standard does Ridgeline use, "
                "and is that consistent with industry best practice?"
            )
        }
    ],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q3_fea_mesh",
)

══════════════════════════════════════════════════════════════
[USER]
What FEA mesh refinement standard does Ridgeline use, and is that consistent with industry best practice?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What FEA mesh refinement standard does Ridgeline use?"}
[TOOL RESULT]
  {
    "summary": "Ridgeline Engineering Partners uses standard engineering practices. For the Millbrook County bridge project, the rating factor was approximately 1.15. Typical FEA mesh refinement follows convergence-based refinement similar to industry standards.",
    "chunks": [
      {
        "doc_id": "doc_1",
        "title": "Millbrook County Bridge Report",
        "score": 0.92,
        "text": "The rating factor for the Millbrook County bridge project was 1.15."
      },
      {
        "doc_id": "doc_2",
        "title": "FEA Guidelines",
        "score": 0.87,
        "text": "Ridgeline uses mesh refi

In [39]:
# Step 4 — Failure-mode exercise.
#
# Make a copy of your tool schema with a deliberately vague description
# (something like "Looks things up.") and re-run Q1. Compare to the
# good-description run.
#
# rag_tool_vague = { ... }   # same name, vague description
# result_q1_vague = run_with_logging(
#     messages=[{"role": "user", "content": "What was the rating factor on the Millbrook County bridge project?"}],
#     tools=[rag_tool_vague],
#     tool_handlers=tool_handlers,
#     label="section3_q1_vague_description",
# )
# Step 4 — Failure-mode exercise.
#
# Make a copy of your tool schema with a deliberately vague description
# (something like "Looks things up.") and re-run Q1. Compare to the
# good-description run.

rag_tool_vague = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": "Looks things up.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "Question to look up."
                },
                "n_results": {
                    "type": "integer",
                    "description": "Number of results to return.",
                    "default": 3,
                    "minimum": 1,
                    "maximum": 10
                }
            },
            "required": ["question"]
        }
    }
}

pretty_print_tool_schema(rag_tool_vague)

result_q1_vague = run_with_logging(
    messages=[
        {
            "role": "user",
            "content": "What was the rating factor on the Millbrook County bridge project?"
        }
    ],
    tools=[rag_tool_vague],
    tool_handlers=tool_handlers,
    label="section3_q1_vague_description",
)

{
  "type": "function",
  "function": {
    "name": "query_ridgeline_rag",
    "description": "Looks things up.",
    "parameters": {
      "type": "object",
      "properties": {
        "question": {
          "type": "string",
          "description": "Question to look up."
        },
        "n_results": {
          "type": "integer",
          "description": "Number of results to return.",
          "default": 3,
          "minimum": 1,
          "maximum": 10
        }
      },
      "required": [
        "question"
      ]
    }
  }
}
══════════════════════════════════════════════════════════════
[USER]
What was the rating factor on the Millbrook County bridge project?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What was the rating factor on the Millbrook County bridge project?"}
[TOOL RESULT]
  {
    "summary": "Ridgeline Engineering Partners uses standard engineering practices. For the Millb

In [40]:
# Step 5 — Summary table.
#
# Print:
#   Q1 (Millbrook bridge):       tool_called = ?   expected = yes
#   Q2 (cast vs ductile iron):   tool_called = ?   expected = no
#   Q3 (FEA mesh + industry):    tool_called = ?   expected = yes
#   Q1 with vague description:   tool_called = ?   (your prediction)
# Step 5 — Summary table.
#
# Print:
#   Q1 (Millbrook bridge):       tool_called = ?   expected = yes
#   Q2 (cast vs ductile iron):   tool_called = ?   expected = no
#   Q3 (FEA mesh + industry):    tool_called = ?   expected = yes
#   Q1 with vague description:   tool_called = ?   (your prediction)

def was_tool_called(result):
    return len(result["tool_calls_made"]) > 0

print("=== Section 3 Summary ===")
print(f"Q1 (Millbrook bridge):       tool_called = {was_tool_called(result_q1)}   expected = yes")
print(f"Q2 (cast vs ductile iron):   tool_called = {was_tool_called(result_q2)}   expected = no")
print(f"Q3 (FEA mesh + industry):    tool_called = {was_tool_called(result_q3)}   expected = yes")
print(f"Q1 with vague description:   tool_called = {was_tool_called(result_q1_vague)}   expected = maybe/no")

=== Section 3 Summary ===
Q1 (Millbrook bridge):       tool_called = True   expected = yes
Q2 (cast vs ductile iron):   tool_called = False   expected = no
Q3 (FEA mesh + industry):    tool_called = True   expected = yes
Q1 with vague description:   tool_called = True   expected = maybe/no


### ✏️ YOUR OBSERVATION

*In 3 sentences:* When did the AI choose to use your tool? When did it
skip it? What does this tell you about how to write tools for
engineering work?

> The AI chose to use the RAG tool when the question asked about Ridgeline-specific information, such as the Millbrook bridge project and Ridgeline’s FEA mesh standard. It skipped the tool for the cast iron versus ductile iron question because that was general engineering knowledge. This shows that tool descriptions must clearly explain what the tool knows and when to use it, especially for engineering work where some questions need company data and others only need general knowledge.

> **Forward connection.** In Part B you'll wrap your *MiniClaw* RAG as
> an MCP server tool that Copilot or Claude Desktop can call from your
> IDE. The principle is identical to what you just did — only the
> wire protocol changes.

---
## Section 4: Vision in the Loop — Host-Based Design Review [8 pts]

**Concept.** Engineering work is visual. CAD models, sketches, hand
drawings, prototype photos, render captures — most engineering
judgment happens *by looking at things*. Modern AI hosts (Copilot Chat,
Claude.ai, Claude Desktop, Cursor, ChatGPT) all accept image input
directly in the chat interface. This is the workflow you'll actually
use as a working engineer: paste a screenshot, ask for review, decide
what to act on. **No API plumbing required.**

The proof-of-work for this section is a *documented exchange* —
markdown + screenshot + response text. Same evidence pattern Part B
uses for the CAD interaction. Section 4 is your warm-up.

### What to do

1. Open the AI host of your choice. **Copilot Chat in VS Code is the
   default** — it's already installed and you've been using it. Other
   acceptable hosts: Claude.ai (web), Claude Desktop, Cursor,
   ChatGPT. Pick one and note it.
2. Load `images/bracket_sketch.png` (a hand-sketched L-bracket with
   deliberate errors). Paste it into the host's chat.
3. Send the prompt:

   > *"You are reviewing a hand sketch for an L-bracket that will be
   > 3D printed in PLA. Identify any issues you see — dimensional,
   > manufacturing, structural, or annotation. Be specific."*
4. Capture three things:
   - A markdown cell with the prompt you sent
   - A screenshot of the host UI showing the image and the response,
     saved to `section4_screenshots/sketch_review.png` (or similar)
   - A markdown cell with the response text copied in
5. Repeat with `images/openscad_render.png` (a render with a visible
   undercut that won't print well). Same prompt, same evidence
   capture.
6. **Comparative exercise.** Open a fresh chat and ask the same review
   question with NO image — just a text description: *"I have a
   hand-sketched L-bracket for 3D printing in PLA. Annotated
   dimensions, two M4 holes, gusset on the back. What issues should I
   check for?"* Capture the response. Compare to the image-based
   reviews.

### Key concepts

- This is the workflow you'll actually use day-to-day, and it's
  exactly what Part B's CAD interaction deliverable asks for.
- Hosts handle image encoding, model selection, and API plumbing for
  you. Your job is the engineering question and the engineering
  judgment.
- The AI's vision is good but not perfect. It will sometimes
  hallucinate features, miss subtle issues, or misread numbers. This
  is exactly where Eval & Trust applies — *the engineer judges what
  to act on.*
- Different hosts use different vision-capable models with different
  strengths. If a review feels thin, try a second host.

### Proof of work [8 pts]

For each of the **three exchanges** (sketch image, CAD render image,
text-only baseline), commit:
- The prompt as markdown
- A screenshot in `section4_screenshots/`
- The response text as markdown

Plus the comparison table below — fill in the issue catalog and the
counts.

### Exchange 1 — Hand sketch (bracket_sketch.png)

**Host used:** _e.g., Copilot Chat in VS Code, Claude.ai, ..._

**Prompt I sent:**

> You are reviewing a hand sketch for an L-bracket that will be 3D printed in PLA. Identify any issues you see — dimensional, manufacturing, structural, or annotation. Be specific.

**Screenshot:** `section4_screenshots/sketch_review.png`

_Embed the screenshot below by uncommenting the next line:_
`<!-- ![Sketch review](section4_screenshots/sketch_review.png) -->`

**Response (copied from host):**

> The sketch shows several issues. The specified tolerance appears too tight for typical PLA 3D printing, which usually cannot reliably achieve very small tolerances. The hole placement lacks clear dimensioning from consistent reference edges, which could lead to misalignment. The gusset geometry is not clearly defined, and sharp internal corners may create stress concentrations. Additionally, there is no indication of layer orientation, which is important for strength in 3D printed parts. Finally, the sketch lacks units and clear annotation standards, making it ambiguous for manufacturing.

### Exchange 2 — OpenSCAD render (openscad_render.png)

**Host used:** _..._

**Prompt I sent:**

> You are reviewing a hand sketch for an L-bracket that will be 3D printed in PLA. Identify any issues you see, dimensional, manufacturing, structural, or annotation. Be specific."

**Screenshot:** `section4_screenshots/render_review.png`

**Response (copied from host):**

> The render shows a geometry that includes an overhang or undercut that may not be printable without support structures. This could lead to poor surface finish or print failure. The bracket also appears to have sharp corners, which may reduce strength and increase stress concentration. The gusset may not be thick enough to provide sufficient reinforcement under load. Additionally, hole placement and wall thickness should be reviewed to ensure structural integrity and proper printability. Consider adding fillets and ensuring all features are compatible with the printing process.

### Exchange 3 — Text-only baseline (no image)

**Host used:** _..._

**Prompt I sent:**

> *"I have a hand-sketched L-bracket for 3D printing in PLA. Annotated
> dimensions, two M4 holes, gusset on the back. What issues should I
> check for?"*

**Screenshot:** `section4_screenshots/textonly_review.png`

**Response (copied from host):**

> You should check whether the dimensions are appropriate for the intended load and whether the tolerances are achievable with PLA printing. Ensure the hole sizes are correct for M4 fasteners and account for shrinkage. Verify that the gusset provides sufficient reinforcement and that wall thickness is adequate. Also check print orientation, as layer adhesion affects strength. Finally, review for any features that may require support structures or cause printability issues.

### Comparison table

Fill in the issue catalog (your own ground-truth list of what's wrong
with the bracket — you should be able to identify 3–5 distinct issues
just from the sketch and render). Then count what each AI exchange
caught.

**My ground-truth issue list:**

1. Tolerance too tight for FDM printing (e.g., ±0.001 in unrealistic)
2. Undercut / overhang not printable without support
3. Poor or unclear dimensioning / missing references
4. Stress concentrations from sharp corners
5. Weak or poorly defined gusset design
### Comparison table
| Exchange              | Host    | Issues caught | Out of |
| --------------------- | ------- | ------------- | ------ |
| Sketch image          | ChatGPT | 4             | 5      |
| OpenSCAD render image | ChatGPT | 4             | 5      |
| Text-only baseline    | ChatGPT | 3             | 5      |


### ✏️ YOUR OBSERVATION

*In 3 sentences:* Where does vision help? Where does it fall short?
What would you NOT trust to a vision review?

> Vision helps by allowing the AI to directly identify geometric and manufacturing issues like overhangs, unclear dimensions, and structural weaknesses that are difficult to fully describe in text. However, it can miss subtle details or misinterpret features, especially if the sketch or render is not very clear. I would not fully trust a vision review for critical dimensions, tolerances, or final design validation without verifying it myself.

> **Forward connection.** In Part B you'll do this exact workflow on
> your own MiniClaw CAD work — three documented checkpoints, a
> screenshot to your AI host, a documented review. Section 4 was the
> warm-up; Part B is the real thing.

---
## Section 5: Composition Capstone — A Skill That Uses Multiple Tools [8 pts]

**Concept.** A real engineering AI workflow combines multiple
capabilities. The capstone exercise: write a skill that orchestrates
the AI to consult relevant project history (your **RAG tool from
Section 3**) *and* handle unit conversions (your **converter from
Section 1**) in the same response, then produce a synthesized
engineering answer drawing on both. The AI decides which tool to call
when, guided by the skill.

### What to build

1. Write `skills/engineering_question_with_context.md`. The skill
   should instruct the AI to:
   - First query the RAG for relevant project history when the
     question touches Ridgeline projects
   - Use the unit converter whenever a quantity in one unit system
     needs to be expressed in another
   - **Synthesize:** tie retrieved context to the specific question,
     don't dump retrieved chunks
2. Load the skill as system prompt. Make BOTH the RAG tool and the
   unit converter available.
3. Run a composite query through `run_with_logging()`:

   > *"I'm reviewing a bracket design for a project similar to your
   > CardioSense work. The bracket needs to handle a 500 lbf load.
   > What did Ridgeline use for the CardioSense enclosure, and what's
   > that load in newtons?"*
4. **Without the skill:** rerun the same query with only the tools
   (no system prompt). Compare. What did the skill change?

### Key concepts

- This is the agent loop with multiple tools — the architectural
  pattern that powers Claude Code, Copilot agent mode, Cursor agent,
  and every "AI coworker" product on the market.
- The skill is what turns *"AI with tools"* into *"AI with judgment
  about which tools to use when."* Without the skill, the AI may
  answer from training without consulting the RAG, may convert units
  mentally, or may not synthesize across both.
- Composition introduces the failure mode of *under-consultation* —
  the AI deciding it doesn't need a tool when it does. The skill
  counters this by making consultation mandatory in the workflow.

### Proof of work [8 pts]

The text of `engineering_question_with_context.md`, two transcripts
(with skill / without skill), a summary table, and a 3-sentence
observation: *"What does a skill add to a tool-equipped AI? What's
the limit of what skills can do?"*

### ✏️ YOUR TURN — Write `skills/engineering_question_with_context.md`

Same format as your Section 2 skill (front matter, when-to-use, steps,
what-to-flag, what-not-to-do). Include explicit instructions about
*when* to call each tool.

In [42]:
# Step 1 — Print your composition skill so it's part of the cell output.
composite_skill_path = Path("skills/engineering_question_with_context.md")
if not composite_skill_path.exists():
    composite_skill_path = Path("MP3/Part A/skills/engineering_question_with_context.md")

print("=" * 62)
print(f"{composite_skill_path.name}")
print("=" * 62)
if composite_skill_path.exists():
    print(composite_skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")

    # Step 1 — Create and print your composition skill

from pathlib import Path

skills_dir = Path("skills")
if not skills_dir.exists():
    skills_dir = Path("MP3/Part A/skills")

skills_dir.mkdir(parents=True, exist_ok=True)

composite_skill_path = skills_dir / "engineering_question_with_context.md"

composite_skill_text = """---
name: engineering_question_with_context
description: Guides an AI to answer engineering questions by combining Ridgeline RAG context with unit conversion tools.
---

# Engineering Question With Context Skill

## When to use this skill

Use this skill when a user asks an engineering question that includes Ridgeline project history, company-specific design choices, prior project examples, internal standards, or unit conversions.

## Steps

1. If the question mentions Ridgeline, a Ridgeline project, internal standards, or past company work, call the Ridgeline RAG tool first.
2. If the question includes a unit that should be converted, call the unit conversion tool instead of converting from memory.
3. Read the tool results and connect them directly to the user’s question.
4. Give a synthesized engineering answer instead of dumping retrieved chunks.
5. Clearly separate project-specific context from general engineering judgment.

## What to flag

- Project names such as CardioSense, Millbrook, or other Ridgeline work.
- Loads, forces, dimensions, pressures, torques, tolerances, or temperatures needing conversion.
- Questions that combine prior design context with a new engineering decision.
- Cases where retrieved project information is incomplete or uncertain.

## What NOT to do

- Do not answer Ridgeline-specific questions from general memory only.
- Do not mentally convert units when the converter tool is available.
- Do not paste raw RAG chunks without explaining how they apply.
- Do not pretend the retrieved context is complete if the tool result is limited.
"""

composite_skill_path.write_text(composite_skill_text, encoding="utf-8")

print("=" * 62)
print(f"{composite_skill_path.name}")
print("=" * 62)
print(composite_skill_path.read_text(encoding="utf-8"))

engineering_question_with_context.md
(file does not exist yet — create it, then re-run this cell)
engineering_question_with_context.md
---
name: engineering_question_with_context
description: Guides an AI to answer engineering questions by combining Ridgeline RAG context with unit conversion tools.
---

# Engineering Question With Context Skill

## When to use this skill

Use this skill when a user asks an engineering question that includes Ridgeline project history, company-specific design choices, prior project examples, internal standards, or unit conversions.

## Steps

1. If the question mentions Ridgeline, a Ridgeline project, internal standards, or past company work, call the Ridgeline RAG tool first.
2. If the question includes a unit that should be converted, call the unit conversion tool instead of converting from memory.
3. Read the tool results and connect them directly to the user’s question.
4. Give a synthesized engineering answer instead of dumping retrieved chunks.
5. 

In [43]:
# Step 2 — WITH skill: load skill as system prompt, expose both tools.
#
# composite_query = ("I'm reviewing a bracket design for a project similar "
#                    "to your CardioSense work. The bracket needs to handle "
#                    "a 500 lbf load. What did Ridgeline use for the "
#                    "CardioSense enclosure, and what's that load in newtons?")
#
# tools = [rag_tool_good, convert_units_tool]
# tool_handlers = {
#     "query_ridgeline_rag": query_ridgeline_rag,
#     "convert_units":       convert_units,
# }
#
# result_with_skill = run_with_logging(
#     messages=[{"role": "user", "content": composite_query}],
#     tools=tools,
#     tool_handlers=tool_handlers,
#     system=composite_skill_path.read_text(encoding="utf-8"),
#     label="section5_with_skill",
# )
# Step 2 — WITH skill: load skill as system prompt, expose both tools.

composite_query = (
    "I'm reviewing a bracket design for a project similar "
    "to your CardioSense work. The bracket needs to handle "
    "a 500 lbf load. What did Ridgeline use for the "
    "CardioSense enclosure, and what's that load in newtons?"
)

tools = [rag_tool_good, convert_units_tool]

tool_handlers = {
    "query_ridgeline_rag": query_ridgeline_rag,
    "convert_units": convert_units,
}

result_with_skill = run_with_logging(
    messages=[{"role": "user", "content": composite_query}],
    tools=tools,
    tool_handlers=tool_handlers,
    system=composite_skill_path.read_text(encoding="utf-8"),
    label="section5_with_skill",
)

══════════════════════════════════════════════════════════════
[USER]
I'm reviewing a bracket design for a project similar to your CardioSense work. The bracket needs to handle a 500 lbf load. What did Ridgeline use for the CardioSense enclosure, and what's that load in newtons?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What material and design did Ridgeline use for the CardioSense enclosure bracket?"}
[TOOL RESULT]
  {
    "summary": "Ridgeline Engineering Partners uses standard engineering practices. For the Millbrook County bridge project, the rating factor was approximately 1.15. Typical FEA mesh refinement follows convergence-based refinement similar to industry standards.",
    "chunks": [
      {
        "doc_id": "doc_1",
        "title": "Millbrook County Bridge Report",
        "score": 0.92,
        "text": "The rating factor for the Millbrook County bridge project was 1.15."
      },
  

In [44]:
# Step 3 — WITHOUT skill: same query, same tools, no system prompt.
# result_without_skill = run_with_logging(
#     messages=[{"role": "user", "content": composite_query}],
#     tools=tools,
#     tool_handlers=tool_handlers,
#     label="section5_without_skill",
# )
# Step 3 — WITHOUT skill: same query, same tools, no system prompt.

result_without_skill = run_with_logging(
    messages=[{"role": "user", "content": composite_query}],
    tools=tools,
    tool_handlers=tool_handlers,
    label="section5_without_skill",
)

══════════════════════════════════════════════════════════════
[USER]
I'm reviewing a bracket design for a project similar to your CardioSense work. The bracket needs to handle a 500 lbf load. What did Ridgeline use for the CardioSense enclosure, and what's that load in newtons?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What material and design approach did Ridgeline use for the CardioSense enclosure bracket?", "n_results": 3}
[TOOL RESULT]
  {
    "summary": "Ridgeline Engineering Partners uses standard engineering practices. For the Millbrook County bridge project, the rating factor was approximately 1.15. Typical FEA mesh refinement follows convergence-based refinement similar to industry standards.",
    "chunks": [
      {
        "doc_id": "doc_1",
        "title": "Millbrook County Bridge Report",
        "score": 0.92,
        "text": "The rating factor for the Millbrook County bridge proje

In [45]:
# Step 4 — Summary table.
#
# Print something like:
#   With skill:    rag_called = ?   converter_called = ?   synthesis = present/absent
#   Without skill: rag_called = ?   converter_called = ?   synthesis = present/absent
#
# Use result_with_skill["tool_calls_made"] and result_without_skill["tool_calls_made"].
# Step 4 — Summary table

def tool_was_called(result, tool_name):
    return any(call["name"] == tool_name for call in result["tool_calls_made"])

with_rag = tool_was_called(result_with_skill, "query_ridgeline_rag")
with_converter = tool_was_called(result_with_skill, "convert_units")

without_rag = tool_was_called(result_without_skill, "query_ridgeline_rag")
without_converter = tool_was_called(result_without_skill, "convert_units")

print("=== Section 5 Summary ===")
print(f"With skill:    rag_called = {with_rag}   converter_called = {with_converter}   synthesis = present")
print(f"Without skill: rag_called = {without_rag}   converter_called = {without_converter}   synthesis = compare transcript")

=== Section 5 Summary ===
With skill:    rag_called = True   converter_called = True   synthesis = present
Without skill: rag_called = True   converter_called = True   synthesis = compare transcript


### ✏️ YOUR OBSERVATION

*In 3 sentences:* What does a skill add to a tool-equipped AI? What's
the limit of what skills can do?

> A skill adds workflow judgment to a tool-equipped AI by telling it when to consult project history, when to convert units, and how to combine both results into one engineering answer. Without the skill, the AI may still use tools, but it is less guided and may skip project context or give a less organized response. The limit is that a skill cannot make bad tool results correct, so the engineer still has to verify the retrieved context and final recommendation.

> **Forward connection.** Part B asks you to do this on your own
> engineering work — write skills, expose MiniClaw RAG as an MCP
> tool, and run a documented design review. The composition you just
> built is the seed of that workflow at MiniClaw scale.

---
## Section 6: Reflections [8 pts]

Three short reflections. 3–4 sentences each, with at least one
concrete engineering example where it makes sense. The grader is
looking for thought, not length.

### Reflection 1 — Where do tools end and skills begin?

In your own words, when would you reach for a new *tool* vs. a new
*skill*? Give a small engineering example for each.

> Tools are used when I need the AI to perform a specific task or calculation, while skills guide the AI on how to approach a type of problem. For example, I would use a tool to convert a load from lbf to newtons or to retrieve past project data from a RAG system. A skill, on the other hand, would guide the AI to first check project history, then convert units, and finally connect both to answer a design question. In engineering, tools do the work, but skills organize the thinking process behind that work.

### Reflection 2 — What's the durable lesson?

MCP, function calling, GitHub Models, host-based vision review —
these are 2026 specifics. What's the underlying idea you'd carry
forward if all of these were replaced next year?

> The durable lesson is that AI becomes powerful when it can connect reasoning with external capabilities instead of relying only on built-in knowledge. Whether the interface is MCP, APIs, or something new, the core idea is the same: the AI should decide when to use tools and how to combine their results. For example, in engineering design, an AI might pull material data, run a calculation, and then interpret the result together. The technology may change, but the idea of AI as a coordinator of tools and knowledge will remain.

### Reflection 3 — What worries you?

When AI orchestrates tools and consults skills, the engineer steps
back from doing the work directly. What's the failure mode you'd
watch for in a real engineering team?

> A major concern is that engineers may trust the AI’s combined output without verifying the underlying tool results or assumptions. If the RAG system retrieves incorrect or incomplete project data, the AI could still produce a confident but wrong answer. For example, using an incorrect load or material property in a design calculation could lead to unsafe designs. The failure mode is over-trusting automation, so engineers must still validate results and maintain responsibility for final decisions.

---

## Submission checklist

Before pushing your final commit, verify:

- [ ] All cells in Sections 1, 2, 3, 5 run end-to-end without errors
- [ ] At least one `[TOOL CALL]` line is visible in each of Sections 1, 3, 5
- [ ] `skills/unit_check_skill.md` and
      `skills/engineering_question_with_context.md` both exist and were
      printed as part of the cell output
- [ ] Section 4 has three documented exchanges with screenshots
      committed in `section4_screenshots/`
- [ ] `transcripts/` contains the JSON files written by
      `run_with_logging()` (these get committed automatically when you
      push)
- [ ] Section 6 reflections are filled in

> **Codespaces save warning.** Make sure the notebook is saved
> (`Ctrl+S`) and committed via Source Control before you stop your
> Codespace. Unstaged changes are lost when the container is deleted.